In [0]:
from pyspark.sql.functions import avg, round, count, year, date_format

# Lê o Silver
df_gold = spark.table("playgroundlearn.combustivel_brasil.silver_gasolina")

# 1 - Preço médio por estado, produto e ano
gold_estado = df_gold.groupBy("estado_sigla", "produto",
    year("data_coleta").alias("ano")) \
    .agg(
        round(avg("valor_venda"), 4).alias("preco_medio_venda"),
        round(avg("valor_compra"), 4).alias("preco_medio_compra"),
        round(avg("margem"), 4).alias("margem_media"),
        count("*").alias("total_registros")
    ).orderBy("ano", "estado_sigla")

# 2 - Preço médio por bandeira, produto e ano
gold_bandeira = df_gold.groupBy("bandeira", "produto",
    year("data_coleta").alias("ano")) \
    .agg(
        round(avg("valor_venda"), 4).alias("preco_medio_venda"),
        round(avg("margem"), 4).alias("margem_media"),
        count("*").alias("total_postos")
    ).orderBy("ano", "bandeira")

# 3 - Evolução mensal por produto
gold_mensal = df_gold.groupBy(
    date_format("data_coleta", "yyyy-MM").alias("ano_mes"),
    "produto"
).agg(
    round(avg("valor_venda"), 4).alias("preco_medio_venda"),
    round(avg("margem"), 4).alias("margem_media")
).orderBy("ano_mes")

# Salva as três tabelas Gold
gold_estado.write.format("delta").mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("playgroundlearn.combustivel_brasil.gold_preco_estado")

gold_bandeira.write.format("delta").mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("playgroundlearn.combustivel_brasil.gold_preco_bandeira")

gold_mensal.write.format("delta").mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("playgroundlearn.combustivel_brasil.gold_evolucao_mensal")

print("Gold salvo com sucesso!")

Gold salvo com sucesso!
